# F-K compare: one file with an earthquake vs one quiet file

Reads two 60-s continuous SAFOD DAS files — one containing a known M~2 earthquake near Parkfield, one from a quiet week — and shows what the raw wavefield looks like in (f, k) space for each.

No filtering yet. Just visualization to see what's there.

Run cell-by-cell. Should take ~1-2 min total.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make DASutils importable in JupyterLab — same path the sbatch scripts use.
sys.path.insert(0, '/home/groups/ettore88/nberrios/safod_das_git/DAS-utilities/python')
import DASutils

In [ ]:
# ---------------- Parameters ----------------
CSV = '/oak/stanford/groups/ettore88/data/SAFOD/SAFODAS1-harddrive-transfer/SAFOD_2024_2025.csv'
DATA_ROOT_OLD = '/oak/stanford/groups/ettore88/data/SAFODAS1-harddrive-transfer'
DATA_ROOT_NEW = '/oak/stanford/groups/ettore88/data/SAFOD/SAFODAS1-harddrive-transfer'

# Channel slice — same as the rest of the sanity pipeline.
CH_START = 150
CH_END   = 800

# The earthquake to use. M2.02 Md, 30 km of Parkfield, in the noisy week 2024-06-04..2024-06-10.
EQ_TIME = pd.Timestamp('2024-06-07T21:48:23.430000', tz='UTC')
EQ_MAG  = 2.02
EQ_MAG_TYPE = 'Md'

# Quiet window — 2024-10-23 20:00 UTC, in a week with 0 M>=1 events in catalog.
QUIET_DATE = '2024-10-23'
QUIET_HOUR = 20

# Pre-bandpass on read.
READ_FMIN, READ_FMAX = 0.05, 24.0

# Reference apparent velocities to overlay on the F-K plot.
REF_VELS = {
    'P body (3200 m/s)':  3200.0,
    'S body (1600 m/s)':  1600.0,
    'Tube (~1500 m/s)':   1500.0,
    'Surface (~500 m/s)': 500.0,
}

## Load manifest and find the two files

In [ ]:
def normalize_path(p):
    p = str(p)
    if os.path.exists(p):
        return p
    if p.startswith(DATA_ROOT_OLD):
        alt = p.replace(DATA_ROOT_OLD, DATA_ROOT_NEW, 1)
        if os.path.exists(alt):
            return alt
    return p

db = pd.read_csv(CSV, sep=r'\s+').drop_duplicates()
db = db[db['nSamples'] == 30000].reset_index(drop=True)
db['startTime_dt'] = pd.to_datetime(db['startTime'], errors='coerce', utc=True)
db['endTime_dt']   = pd.to_datetime(db['endTime'],   errors='coerce', utc=True)
db = db.dropna(subset=['startTime_dt', 'endTime_dt']).reset_index(drop=True)
db['file_norm'] = db['file'].map(normalize_path)
db = db[db['file_norm'].map(os.path.exists)].reset_index(drop=True)
print(f'Loaded {len(db)} continuous (nSamples=30000) files from manifest.')

In [ ]:
# Find the 60-s file whose [startTime, endTime] contains EQ_TIME.
eq_match = db[(db['startTime_dt'] <= EQ_TIME) & (db['endTime_dt'] >= EQ_TIME)]
assert len(eq_match) > 0, f'No file contains {EQ_TIME}'
eq_row = eq_match.iloc[0]
print(f'EQ file:  {eq_row["file_norm"]}')
print(f'  window: {eq_row["startTime_dt"]} to {eq_row["endTime_dt"]}')
print(f'  EQ at offset {(EQ_TIME - eq_row["startTime_dt"]).total_seconds():.2f} s within the file')

quiet_match = db[
    (db['startTime_dt'].dt.strftime('%Y-%m-%d') == QUIET_DATE)
    & (db['startTime_dt'].dt.hour == QUIET_HOUR)
]
assert len(quiet_match) > 0, f'No quiet file for {QUIET_DATE} {QUIET_HOUR:02d}:XX UTC'
quiet_row = quiet_match.iloc[0]
print(f'Quiet file: {quiet_row["file_norm"]}')
print(f'  window: {quiet_row["startTime_dt"]} to {quiet_row["endTime_dt"]}')

## Read both files

In [ ]:
def read_file(path):
    DAS, info = DASutils.readFile_HDF(
        [path], READ_FMIN, READ_FMAX, verbose=0,
        preproc=True, diff=True, taper=False,
        desampling=True, nChbuffer=900, system='OptaSense',
    )
    return DAS[CH_START:CH_END, :].astype(np.float64, copy=False), info

print('Reading EQ file...')
X_eq, info_eq = read_file(eq_row['file_norm'])
fs_eq = float(info_eq['fs'])
print(f'  shape={X_eq.shape}, fs={fs_eq:.2f} Hz')

print('Reading quiet file...')
X_quiet, info_quiet = read_file(quiet_row['file_norm'])
fs_quiet = float(info_quiet['fs'])
print(f'  shape={X_quiet.shape}, fs={fs_quiet:.2f} Hz')

dz = float(eq_row['dCh'])
print(f'dz = {dz:.4f} m')

## Raw time-distance plots

Plotted side by side. The EQ should be visible as a bright burst against the noise floor.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, X, fs, title in [
    (axs[0], X_eq, fs_eq,
     f'EQ: M{EQ_MAG} {EQ_MAG_TYPE} at {EQ_TIME.strftime("%Y-%m-%d %H:%M:%S")} UTC'),
    (axs[1], X_quiet, fs_quiet,
     f'Quiet: {QUIET_DATE} {QUIET_HOUR:02d}:XX UTC'),
]:
    nch, npts = X.shape
    t = np.arange(npts) / fs
    vlim = np.percentile(np.abs(X), 98)
    ax.imshow(
        X, aspect='auto', origin='upper', cmap='RdBu_r',
        vmin=-vlim, vmax=vlim,
        extent=[t[0], t[-1], CH_END - 1, CH_START],
    )
    ax.set_xlabel('Time (s)')
    ax.set_title(title, fontsize=10)
axs[0].set_ylabel('Channel')
plt.tight_layout()
plt.show()

## Compute F-K spectra

In [ ]:
def compute_fk(X, fs, dz):
    nch, npts = X.shape
    win_t = np.hanning(npts)[None, :]
    win_x = np.hanning(nch)[:, None]
    Xw = X * win_t * win_x
    F  = np.fft.rfft(Xw, axis=1)
    FK = np.fft.fftshift(np.fft.fft(F, axis=0), axes=0)
    f  = np.fft.rfftfreq(npts, d=1.0/fs)
    k  = np.fft.fftshift(np.fft.fftfreq(nch, d=dz))
    return f, k, np.abs(FK)**2

f_eq, k_eq, FK_eq = compute_fk(X_eq, fs_eq, dz)
f_q,  k_q,  FK_q  = compute_fk(X_quiet, fs_quiet, dz)
print('F-K shapes:', FK_eq.shape, FK_q.shape)

## F-K plots side by side

Body-wave energy shows up as a steep (near-vertical) ridge centered at k=0. Tube/surface modes show up as shallow ridges going outward from k=0. If the EQ injects strong body-wave energy, the left panel should show brighter near-vertical structure than the right panel.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

for ax, f, k, FK, title in [
    (axs[0], f_eq, k_eq, FK_eq, f'F-K, EQ window ({EQ_TIME.strftime("%Y-%m-%d %H:%M:%S")} UTC)'),
    (axs[1], f_q,  k_q,  FK_q,  f'F-K, quiet window ({QUIET_DATE} {QUIET_HOUR:02d}:XX UTC)'),
]:
    f_max_disp = 25.0
    fmask = f <= f_max_disp
    f_d = f[fmask]
    P = 10.0 * np.log10(np.maximum(FK[:, fmask], 1e-30))
    pcm = ax.pcolormesh(
        k, f_d, P.T, shading='auto', cmap='magma',
        vmin=np.percentile(P, 70), vmax=np.percentile(P, 99.5),
    )
    for label, v in REF_VELS.items():
        f_line = np.linspace(0.1, f_max_disp, 200)
        k_line = f_line / v
        ax.plot( k_line, f_line, '--', lw=0.8, label=label)
        ax.plot(-k_line, f_line, '--', lw=0.8, color=ax.lines[-1].get_color())
    ax.set_xlim(-0.05, 0.05)
    ax.set_ylim(0, f_max_disp)
    ax.axhspan(5, 20, color='cyan', alpha=0.15)
    ax.set_xlabel('k (cycles / m)')
    ax.set_title(title, fontsize=10)
    plt.colorbar(pcm, ax=ax, label='Power (dB)')
axs[0].set_ylabel('Frequency (Hz)')
axs[0].legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.show()

## Save the figure (optional)

In [ ]:
out_dir = '/home/groups/ettore88/nberrios/safod-das/sanity/fk_compare_out'
os.makedirs(out_dir, exist_ok=True)
fig.savefig(os.path.join(out_dir, 'fk_compare_eq_quiet.png'), dpi=150, bbox_inches='tight')
print(f'Saved {out_dir}/fk_compare_eq_quiet.png')